# <center> Méthodes Statistiques de l'Évaluation 

## Préambule

L'objectif est d'appliquer le modèle de Rubin (identifier statistiquement l'effet causal d'une variable sur une autre) afin de mettre en valeur l'implication des biais (sélection et hétérogénéité) et l'implication de la dépendance entre les variables. 

À partir des modèles construits et des bases de données fournies, nous pouvons construire plusieurs métriques : l'ATE (Average Treatment Effect) et l'ATT (Average Treatment Effect on the Treated), Score moyen. L'analyse commence par identifier les sources de biais présents ou non dans nos données, puis nous verrons si l'assignation du traitement est indépendante.

### Librairies

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

### Préparation des données

Les trois jeux de données (`data0`, `data1`, `data2`) sont générés par le script [`chapitre-1-generate-data.py`](../src/chapitre-1-generate-data.py) (exécuter `python src/chapitre-1-generate-data.py` depuis la racine du dépôt avant de lancer ce notebook si le dossier `data/chapitre-1/` est absent). Le processus génératif reproduit les trois scénarios étudiés : absence de biais, biais de sélection, biais de sélection et d'hétérogénéité — voir le [README](../README.md) pour le détail.

In [2]:
df0 = pd.read_csv("../data/chapitre-1/data0.csv", sep=";")
df1 = pd.read_csv("../data/chapitre-1/data1.csv", sep=";")
df2 = pd.read_csv("../data/chapitre-1/data2.csv", sep=";")

In [3]:
for df in [df0, df1, df2]:
    print(df.columns)
    print(df.dtypes)
    print("\n")

Index(['Y0', 'Y1', 'X', 'Treated', 'Y'], dtype='object')
Y0         float64
Y1         float64
X            int64
Treated      int64
Y          float64
dtype: object


Index(['Y0', 'Y1', 'X', 'Treated', 'Y'], dtype='object')
Y0         float64
Y1         float64
X            int64
Treated      int64
Y          float64
dtype: object


Index(['Y0', 'Y1', 'X', 'Treated', 'Y'], dtype='object')
Y0         float64
Y1         float64
X            int64
Treated      int64
Y          float64
dtype: object




In [4]:
for df in [df0, df1, df2]:
    df["Treated"] = df["Treated"].astype("category")

In [5]:
df0

,Y0,Y1,X,Treated,Y
0,10.94,17.61,1,0,10.94
1,12.53,17.02,0,1,17.02
2,12.13,17.72,1,0,12.13
3,11.29,16.36,1,0,11.29
4,14.37,18.64,0,0,14.37
...,...,...,...,...,...
49995,15.76,21.28,1,0,15.76
49996,14.08,20.11,0,0,14.08
49997,13.91,19.90,1,0,13.91
49998,7.95,12.34,1,0,7.95


In [6]:
df1

,Y0,Y1,X,Treated,Y
0,10.56,16.60,0,1,16.60
1,18.70,24.56,0,0,18.70
2,7.80,12.34,1,0,7.80
3,5.78,10.03,1,0,5.78
4,11.85,18.43,1,1,18.43
...,...,...,...,...,...
49995,12.51,18.80,0,1,18.80
49996,7.69,12.43,1,0,7.69
49997,11.20,16.45,0,1,16.45
49998,10.80,16.41,1,0,10.80


In [7]:
df2

,Y0,Y1,X,Treated,Y
0,12.67,14.00,0,0,12.67
1,18.37,19.71,0,0,18.37
2,13.84,16.35,0,0,13.84
3,11.44,12.82,1,0,11.44
4,15.07,16.43,0,0,15.07
...,...,...,...,...,...
49995,10.29,11.14,1,0,10.29
49996,9.61,9.66,1,0,9.61
49997,15.63,17.71,0,0,15.63
49998,12.80,13.00,0,0,12.80


## Analyse

### Modèle de Rubin

Spécification du modèle de Rubin :

$$Y_i = \alpha + \Delta T_i + \omega_i$$

- $Y_i$ : le revenu observé de l'individu $i$
- $\alpha$ : constante, valeur moyenne de $Y$ quand $T_i = 0$ (la moyenne des non-traités)
- $\Delta$ : effet du traitement, estimateur
- $T_i$ : le traitement de l'individu $i$, vaut $0$ (non-traité) ou $1$ (traité)
- $\omega_i$ : le terme d'erreur de l'individu $i$

Explication de $Y_1$, $Y_0$ et $Y$ dans le modèle de Rubin : 

- $Y_i(0)$ : le gain potentiel de l'individu $i$ en l'absence de traitement
- $Y_i(1)$ : le gain potentiel de l'individu $i$ sous traitement

Le revenu observé ($Y_i$) est :

$$Y_i = T_i \times Y_i(1) + (1 - T_i) \times Y_i(0)$$

- Si $T_i = 0$ : on observe $Y_i = Y_i(0)$ et $Y_i(1)$ représente le gain potentiel (non-observable)
- Si $T_i = 1$ : on observe $Y_i = Y_i(1)$ et $Y_i(0)$ représente le gain potentiel (observable avant le traitement)

### Estimations par OLS

In [8]:
model0 = sm.OLS(df0["Y"], sm.add_constant(df0["Treated"])).fit()
print(model0.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.290
Model:                            OLS   Adj. R-squared:                  0.290
Method:                 Least Squares   F-statistic:                 2.045e+04
Date:                Thu, 10 Sep 2026   Prob (F-statistic):               0.00
Time:                        20:50:17   Log-Likelihood:            -1.3535e+05
No. Observations:               50000   AIC:                         2.707e+05
Df Residuals:                   49998   BIC:                         2.707e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         12.8465      0.019    686.381      0.0

In [9]:
model1 = sm.OLS(df1["Y"], sm.add_constant(df1["Treated"])).fit()
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.367
Model:                            OLS   Adj. R-squared:                  0.367
Method:                 Least Squares   F-statistic:                 2.897e+04
Date:                Thu, 10 Sep 2026   Prob (F-statistic):               0.00
Time:                        20:50:17   Log-Likelihood:            -1.3488e+05
No. Observations:               50000   AIC:                         2.698e+05
Df Residuals:                   49998   BIC:                         2.698e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         12.6384      0.019    682.586      0.0

In [10]:
model2 = sm.OLS(df2["Y"], sm.add_constant(df2["Treated"])).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.080
Model:                            OLS   Adj. R-squared:                  0.080
Method:                 Least Squares   F-statistic:                     4366.
Date:                Thu, 10 Sep 2026   Prob (F-statistic):               0.00
Time:                        20:50:17   Log-Likelihood:            -1.3570e+05
No. Observations:               50000   AIC:                         2.714e+05
Df Residuals:                   49998   BIC:                         2.714e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         12.6379      0.019    670.349      0.0

### Composition de ATE (*Average Treatment Effect*) et ATT (*Average Treatment Effect on Treated*)

Calcul de l'effet moyen du traitement ($\Delta^{\text{ATE}}$) et l'effet moyen sur les traités ($\Delta^{\text{ATT}}$)


$$\Delta^{\text{ATE}} = E[Y_i(1) - Y_i(0)] \qquad \text{(effet moyen sur la population)}$$

$$\Delta^{\text{ATT}} = E[Y_i(1) - Y_i(0) \mid T_i = 1] \qquad \text{(effet moyen sur les traités)}$$

$\Delta^{\text{ATT}}$ est aussi le coefficient $\Delta$ de la régression par OLS sans biais

In [11]:
df = [df0, df1, df2]
model = [model0, model1, model2]

for i in range(3):
    print(f"base {i} : ")
    
    ATE = df[i]["Y1"].mean() - df[i]["Y0"].mean()
    ATT = df[i].loc[df[i]["Treated"] == 1, "Y1"].mean() - df[i].loc[df[i]["Treated"] == 1, "Y0"].mean()
    
    print(f"Modèle {i} → ATE : {round(ATE, 2)} | ATT {round(ATT, 2)} \n")

for i in range(3):
    print("Comparaison")
    print(f"base {i} : ")

    ATT = df[i].loc[df[i]["Treated"] == 1, "Y1"].mean() - df[i].loc[df[i]["Treated"] == 1, "Y0"].mean()
    ATT_OLS = model[i].params["Treated"]

    print(f"ATT calculé : {round(ATT, 2)} | ATT OLS : {round(ATT_OLS, 2)}")


base 0 : 
Modèle 0 → ATE : 5.3 | ATT 5.31 

base 1 : 
Modèle 1 → ATE : 5.3 | ATT 5.29 

base 2 : 
Modèle 2 → ATE : 1.2 | ATT 1.36 

Comparaison
base 0 : 
ATT calculé : 5.31 | ATT OLS : 5.36
Comparaison
base 1 : 
ATT calculé : 5.29 | ATT OLS : 6.34
Comparaison
base 2 : 
ATT calculé : 1.36 | ATT OLS : 2.49


### Biais de sélection et d'hétérogénéité

In [12]:
for i in range(3):
    print(f"base {i} : ")
    
    ATE = df[i]["Y1"].mean() - df[i]["Y0"].mean()
    ATT = df[i].loc[df[i]["Treated"] == 1, "Y1"].mean() - df[i].loc[df[i]["Treated"] == 1, "Y0"].mean()

    biais_selection = df[i].loc[df[i]["Treated"]==1, "Y0"].mean() - df[i].loc[df[i]["Treated"]==0, "Y0"].mean()
    naif = ATE + biais_selection
    biais_heterogeneite = ATE - ATT

    if naif == ATE:
        print("biais sélection ➔ ∅")
    else : 
        print(f"biais sélection ➔ {biais_selection}")

    if biais_heterogeneite == 0:
        print("biais hétérogénéité ➔ ∅")
    else : 
        print(f"biais hétérogénéité ➔ {biais_heterogeneite}")
    
    print("\n")

base 0 : 
biais sélection ➔ 0.05348543241876946
biais hétérogénéité ➔ -0.008575099614889226


base 1 : 
biais sélection ➔ 1.0442075550527878
biais hétérogénéité ➔ 0.006031419490495793


base 2 : 
biais sélection ➔ 1.1354420249254442
biais hétérogénéité ➔ -0.15543448009611716




 - Le dataset n°0 n'est pas biaisé → $\Delta^{\text{Naïf}} = \Delta^{\text{ATE}}$
 - Le dataset n°1 contient un biais de sélection → $\Delta^{\text{Naïf}} ≠ \Delta^{\text{ATE}}$
 - Le dataset n°2 contient un biais de sélection et d'hétérogénéité → $\Delta^{\text{Naïf}} ≠ \Delta^{\text{ATE}}$

Ici seul le dataset n°0 recense une expérience randomnisée. 

*N.B.* En réalité, ces informations ne sont pas observables puisque $Y_i(0)$ (le gain potentiel de l'individu $i$ en l'absence de traitement) et $Y_i(1)$ (le gain potentiel sous traitement) ne peuvent pas être observés simultanément pour un même individu.

### Différence moyenne entre population traitée et non-traitée

In [13]:
for i in range(3):
    print(f"base {i} : ")

    Y1_traite = df[i].loc[df[i]["Treated"] ==1, "Y1"].mean()
    Y0_non_traite = df[i].loc[df[i]["Treated"] ==0, "Y0"].mean()

    Y10_diff = Y1_traite - Y0_non_traite

    print(f"Score moyen population traité : {Y1_traite}")
    print(f"Score moyen population non-traité : {Y0_non_traite}")
    print(f"Différence moyenne des groupes : {Y10_diff}", "\n")

base 0 : 
Score moyen population traité : 18.206765083440306
Score moyen population non-traité : 12.846508951406648
Différence moyenne des groupes : 5.360256132033658 

base 1 : 
Score moyen population traité : 18.975241407197736
Score moyen population non-traité : 12.638350471635444
Différence moyenne des groupes : 6.336890935562291 

base 2 : 
Score moyen population traité : 15.130692831397678
Score moyen population non-traité : 12.637850726376117
Différence moyenne des groupes : 2.492842105021561 



En comparant les différences de moyenne, on remarque ici que le biais de sélection et d'hétérogénéité surestiment l'effet moyen du traitement (ATE).

### L'assignement du traitement est-elle indépendante à la variable X ? 

In [14]:
for i in range(3):
    print(f"Base {i} :")
    X_traite = df[i].loc[df[i]["Treated"]==1, "X"].mean()
    X_non_traite = df[i].loc[df[i]["Treated"]==0, "X"].mean()

    X_diff = X_traite - X_non_traite

    print(f"Moyenne X traités : {X_traite}")
    print(f"Moyenne X non-traités : {X_non_traite}")
    print(f"Différence : {X_diff}\n")

Base 0 :
Moyenne X traités : 0.4961489088575096
Moyenne X non-traités : 0.5062340153452686
Différence : -0.010085106487758955

Base 1 :
Moyenne X traités : 0.29818034775576224
Moyenne X non-traités : 0.5685930649661219
Différence : -0.2704127172103597

Base 2 :
Moyenne X traités : 0.2954745694833801
Moyenne X non-traités : 0.5637744902039185
Différence : -0.2682999207205384



 - Dataset n°0, l'assignement du traitement est indépendante conditionnellement à X, la différence de moyenne X = 0. X est équilibré entre le groupe traité et le groupe non-traité

 - Dataset n°1, l'assignement du traitement n'est pas indépendante conditionnellement à X, la différence de moyenne X ≠ 0. X n'est pas équilibré entre le groupe traité et le groupe non-traité
 
 - Dataset n°2, l'assignement du traitement n'est pas indépendante conditionnellement à X, la différence de moyenne X ≠ 0. X n'est pas équilibré entre le groupe traité et le groupe non-traité


### Estimation sur un échantillonnage (10%)

Régression : Y sur T

$$Y_i = \alpha + \Delta T_i + \varepsilon_i$$

Régression bis : Y sur T et X

$$Y_i = \alpha + \Delta T_i + \beta X_i + \varepsilon_i$$

Où :
- $\Delta$ : effet estimé du traitement
- $\beta$ : effet de la variable de contrôle $X$
- $\varepsilon_i$ : terme d'erreur

In [15]:
df0_sub = df0.sample(frac=0.1, replace=False, random_state=42).reset_index(drop=True)
df1_sub = df1.sample(frac=0.1, replace=False, random_state=42).reset_index(drop=True)
df2_sub = df2.sample(frac=0.1, replace=False, random_state=42).reset_index(drop=True)

In [16]:
sub_model0 = sm.OLS(df0_sub["Y"], sm.add_constant(df0_sub["Treated"])).fit()
print(sub_model0.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.305
Model:                            OLS   Adj. R-squared:                  0.305
Method:                 Least Squares   F-statistic:                     2198.
Date:                Thu, 10 Sep 2026   Prob (F-statistic):               0.00
Time:                        20:50:17   Log-Likelihood:                -13479.
No. Observations:                5000   AIC:                         2.696e+04
Df Residuals:                    4998   BIC:                         2.698e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         12.8886      0.059    219.765      0.0

In [17]:
sub_model0_bis = sm.OLS(df0_sub["Y"], sm.add_constant(df0_sub[["Treated", "X"]])).fit()
print(sub_model0_bis.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.540
Model:                            OLS   Adj. R-squared:                  0.540
Method:                 Least Squares   F-statistic:                     2936.
Date:                Thu, 10 Sep 2026   Prob (F-statistic):               0.00
Time:                        20:50:17   Log-Likelihood:                -12447.
No. Observations:                5000   AIC:                         2.490e+04
Df Residuals:                    4997   BIC:                         2.492e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         14.9732      0.063    237.370      0.0

In [18]:
sub_model1 = sm.OLS(df1_sub["Y"], sm.add_constant(df1_sub["Treated"])).fit()
print(sub_model1.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.372
Model:                            OLS   Adj. R-squared:                  0.372
Method:                 Least Squares   F-statistic:                     2960.
Date:                Thu, 10 Sep 2026   Prob (F-statistic):               0.00
Time:                        20:50:17   Log-Likelihood:                -13462.
No. Observations:                5000   AIC:                         2.693e+04
Df Residuals:                    4998   BIC:                         2.694e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         12.6193      0.058    217.192      0.0

In [19]:
sub_model1_bis = sm.OLS(df1_sub["Y"], sm.add_constant(df1_sub[["Treated", "X"]])).fit()
print(sub_model1_bis.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.571
Model:                            OLS   Adj. R-squared:                  0.571
Method:                 Least Squares   F-statistic:                     3328.
Date:                Thu, 10 Sep 2026   Prob (F-statistic):               0.00
Time:                        20:50:17   Log-Likelihood:                -12508.
No. Observations:                5000   AIC:                         2.502e+04
Df Residuals:                    4997   BIC:                         2.504e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         14.9901      0.069    218.047      0.0

In [20]:
sub_model2 = sm.OLS(df2_sub["Y"], sm.add_constant(df2_sub["Treated"])).fit()
print(sub_model2.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.083
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     453.0
Date:                Thu, 10 Sep 2026   Prob (F-statistic):           2.65e-96
Time:                        20:50:17   Log-Likelihood:                -13515.
No. Observations:                5000   AIC:                         2.703e+04
Df Residuals:                    4998   BIC:                         2.705e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         12.6520      0.059    214.397      0.0

In [21]:
sub_model2_bis = sm.OLS(df2_sub["Y"], sm.add_constant(df2_sub[["Treated", "X"]])).fit()
print(sub_model2_bis.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.406
Model:                            OLS   Adj. R-squared:                  0.406
Method:                 Least Squares   F-statistic:                     1711.
Date:                Thu, 10 Sep 2026   Prob (F-statistic):               0.00
Time:                        20:50:17   Log-Likelihood:                -12428.
No. Observations:                5000   AIC:                         2.486e+04
Df Residuals:                    4997   BIC:                         2.488e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         15.1119      0.067    225.834      0.0

L'indépendance de X sur le traitement garantit des groupes homogènes et la présence d'un véritable effet causal du traitement sur la variable Y.

La comparaison des deux modèles (OLS avec et sans X) permet de tester si l'assignation au traitement dépend de X. Si l'ajout de X modifie le coefficient de Treated, alors les individus sont sélectionnés selon X, le traitement n'est pas randomisé. Si le coefficient reste stable, l'assignation est indépendante de X.